In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [3]:
retail_df = pd.read_csv("../datasets/OnlineRetail.csv")

In [4]:
retail_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


* Find if there are any NA values
* Deal with missing values
* How many different customers do we have
* What is the most sold article
* add a column that contains the total value of a row
* create a new dataframe related to invoices only (with customer id, and total value as columns)
* create a new dataframe related to customer only (with a column regarding total spent value, and frequency per invoice)

* Find the invoice with the highest total value
* most frequent customer

* use datetime as an index, and compute the total value spent per month

### NA Values

In [5]:
retail_df.columns.tolist()


['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [6]:
100*retail_df.isna().sum()/len(retail_df)

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

In [7]:
retail_df.loc[retail_df.Description.isna(),"StockCode"]

622        22139
1970       21134
1971       22145
1972       37509
1987      85226A
           ...  
535322     84581
535326     23406
535332     21620
536981     72817
538554     85175
Name: StockCode, Length: 1454, dtype: object

In [8]:
retail_df[retail_df.StockCode == "22139"].Description.iloc[0]

'RETROSPOT TEA SET CERAMIC 11 PC '

In [9]:
def replace_na_description(sc):
    """
    A function that takes as an input a Stockode, and returns a Description
    that is not NA, in case it is not possible, it returns No description available
    
    Ags:
        sc : string related to the StockCode
    
    """
    # we set the default retrun_value to "No description available"
    return_value = "No description available"
    # we create a filter of boolean values representing the elements of the subset Series Description that are not na
    filter_index = retail_df.loc[retail_df.StockCode ==sc,"Description"].notna()
    # If we have at least one True value in the filter_index, we put it in the return_value variable
    if sum(filter_index)>0:
        return_value = retail_df.loc[retail_df.StockCode ==sc,"Description"][filter_index].iloc[0]
    return return_value

In [10]:
replace_na_description("22139")

'RETROSPOT TEA SET CERAMIC 11 PC '

In [11]:
retail_df.loc[retail_df.Description.isna(),"Description"] = retail_df.loc[
    retail_df.Description.isna(),"StockCode"
].apply(replace_na_description)

In [12]:
retail_df.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [13]:
filter_index = retail_df.loc[retail_df.StockCode == "22145","Description"].notna()


retail_df.loc[retail_df.StockCode == "22145","Description"][filter_index].iloc[0]

'CHRISTMAS CRAFT HEART STOCKING '

In [14]:
retail_df.loc[:,"Total_value"] = retail_df.UnitPrice * retail_df.Quantity 

In [15]:
retail_df.Quantity 

0          6
1          6
2          8
3          6
4          6
          ..
541904    12
541905     6
541906     4
541907     4
541908     3
Name: Quantity, Length: 541909, dtype: int64

In [16]:
retail_df.index = pd.to_datetime(retail_df.InvoiceDate)

In [17]:
agg_retail = retail_df.resample("D").agg({"Total_value":"sum","InvoiceNo":"count"})

In [18]:
import plotly.express as px
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True) 

In [19]:
agg_retail.columns

Index(['Total_value', 'InvoiceNo'], dtype='object')

In [21]:
iplot(px.line(agg_retail.loc["2011-03":"2011-09"], y="Total_value",hover_data=["InvoiceNo"]))

In [22]:
retail_df.to_csv("../datasets/processed_retail_dataset.csv")

In [23]:
processed_df = pd.read_csv("../datasets/processed_retail_dataset.csv",index_col="InvoiceDate")

processed_df.index = pd.to_datetime(processed_df.index)

In [24]:
processed_df.index

DatetimeIndex(['2010-12-01 08:26:00', '2010-12-01 08:26:00',
               '2010-12-01 08:26:00', '2010-12-01 08:26:00',
               '2010-12-01 08:26:00', '2010-12-01 08:26:00',
               '2010-12-01 08:26:00', '2010-12-01 08:28:00',
               '2010-12-01 08:28:00', '2010-12-01 08:34:00',
               ...
               '2011-12-09 12:50:00', '2011-12-09 12:50:00',
               '2011-12-09 12:50:00', '2011-12-09 12:50:00',
               '2011-12-09 12:50:00', '2011-12-09 12:50:00',
               '2011-12-09 12:50:00', '2011-12-09 12:50:00',
               '2011-12-09 12:50:00', '2011-12-09 12:50:00'],
              dtype='datetime64[ns]', name='InvoiceDate', length=541909, freq=None)